In [117]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

project_root = Path.cwd().parent
raw_data_dir = project_root / "data" / "raw"

print(raw_data_dir)
print(raw_data_dir.exists())

c:\seizure-detection-ML-project\data\raw
True


In [118]:
chb01_dir = raw_data_dir / "chb01"
edf_files = sorted(chb01_dir.glob("*.edf"))
print("Number of EDF files: ", len(edf_files))

for file in edf_files[:5]:
    print(file.name)


for file in sorted(chb01_dir.iterdir()):
    print(file.name)

Number of EDF files:  3
chb01_01.edf
chb01_02.edf
chb01_03.edf
chb01-summary.txt
chb01_01.edf
chb01_02.edf
chb01_03.edf


In [119]:
summary_file = chb01_dir / "chb01-summary.txt"
summary_text = summary_file.read_text()
print(summary_text[:2000])

Data Sampling Rate: 256 Hz
*************************

Channels in EDF Files:
**********************
Channel 1: FP1-F7
Channel 2: F7-T7
Channel 3: T7-P7
Channel 4: P7-O1
Channel 5: FP1-F3
Channel 6: F3-C3
Channel 7: C3-P3
Channel 8: P3-O1
Channel 9: FP2-F4
Channel 10: F4-C4
Channel 11: C4-P4
Channel 12: P4-O2
Channel 13: FP2-F8
Channel 14: F8-T8
Channel 15: T8-P8
Channel 16: P8-O2
Channel 17: FZ-CZ
Channel 18: CZ-PZ
Channel 19: P7-T7
Channel 20: T7-FT9
Channel 21: FT9-FT10
Channel 22: FT10-T8
Channel 23: T8-P8

File Name: chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0

File Name: chb01_02.edf
File Start Time: 12:42:57
File End Time: 13:42:57
Number of Seizures in File: 0

File Name: chb01_03.edf
File Start Time: 13:43:04
File End Time: 14:43:04
Number of Seizures in File: 1
Seizure Start Time: 2996 seconds
Seizure End Time: 3036 seconds

File Name: chb01_04.edf
File Start Time: 14:43:12
File End Time: 15:43:12
Number of Seizures in File: 1
S

In [120]:
start = summary_text.find("File Name: chb01_01.edf")
end = summary_text.find("File Name: chb01_04.edf")
sample_section = summary_text[start:end]
print(sample_section)

File Name: chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0

File Name: chb01_02.edf
File Start Time: 12:42:57
File End Time: 13:42:57
Number of Seizures in File: 0

File Name: chb01_03.edf
File Start Time: 13:43:04
File End Time: 14:43:04
Number of Seizures in File: 1
Seizure Start Time: 2996 seconds
Seizure End Time: 3036 seconds




In [121]:
recording_blocks = summary_text.split("File Name: ")

print("Number of recording blocks: ", len(recording_blocks))
print(recording_blocks[1])


Number of recording blocks:  43
chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0




In [122]:
recording_blocks = summary_text.split("File Name: ")[1:]
print("Number of recording blocks:", len(recording_blocks))
print(recording_blocks[0])

Number of recording blocks: 42
chb01_01.edf
File Start Time: 11:42:54
File End Time: 12:42:54
Number of Seizures in File: 0




In [123]:
block = recording_blocks[0]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())


print("Filename:", filename)
print("Seizure count:", seizureCount)

Filename: chb01_01.edf
Seizure count: 0


In [124]:
block = recording_blocks[2]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())


print("Filename:", filename)
print("Seizure count:", seizureCount)

Filename: chb01_03.edf
Seizure count: 1


In [125]:
seizure_starts = []
seizure_ends = []
block = recording_blocks[2]
lines = block.splitlines()
filename = lines[0].strip()

for line in lines:
    if line.startswith("Number of Seizures in File: "):
        seizureCount = int(line.split(":")[1].strip())

    if line.startswith("Seizure Start Time: "):
        startTime = line.split(":", 1)[1].strip()
        startTime = startTime.replace(" seconds", "")
        startTime = int(startTime)
        seizure_starts.append(startTime)

    if line.startswith("Seizure End Time: "):
        endTime = line.split(":", 1)[1].strip()
        endTime = endTime.replace(" seconds", "")
        endTime = int(endTime)
        seizure_ends.append(endTime)

print("Filename:", filename)
print("Seizure count:", seizureCount)
print("Starts:", seizure_starts)
print("Ends:", seizure_ends)

seizure_intervals = list(zip(seizure_starts, seizure_ends))
print(seizure_intervals[0])

Filename: chb01_03.edf
Seizure count: 1
Starts: [2996]
Ends: [3036]
(2996, 3036)


In [126]:
seizure_annotations = {}

for block in recording_blocks:
    seizure_starts = []
    seizure_ends = []

    lines = block.splitlines()
    filename = lines[0].strip()

    for line in lines:
        if line.startswith("Number of Seizures in File: "):
            seizure_count = int(line.split(":")[1].strip())

        if line.startswith("Seizure Start Time: "):
            start_time = line.split(":", 1)[1].strip()
            start_time = start_time.replace(" seconds", "")
            start_time = int(start_time)
            seizure_starts.append(start_time)

        if line.startswith("Seizure End Time: "):
            end_time = line.split(":", 1)[1].strip()
            end_time = end_time.replace(" seconds", "")
            end_time = int(end_time)
            seizure_ends.append(end_time)

    seizure_intervals = list(zip(seizure_starts, seizure_ends))
    seizure_annotations[filename] = seizure_intervals

    if len(seizure_intervals) != seizure_count:
        print("Warning:", filename, "seizure count mismatch")

    seizure_annotations[filename] = seizure_intervals


print(seizure_annotations["chb01_01.edf"])
print(seizure_annotations["chb01_02.edf"])
print(seizure_annotations["chb01_03.edf"])


[]
[]
[(2996, 3036)]


In [127]:
for edf_file in edf_files:
    filename = edf_file.name
    intervals = seizure_annotations.get(filename)

    print("filename, intervals:", filename, intervals)

filename, intervals: chb01_01.edf []
filename, intervals: chb01_02.edf []
filename, intervals: chb01_03.edf [(2996, 3036)]


In [128]:
edf_file = edf_files[0]

raw = mne.io.read_raw_edf(edf_file, preload=True)
print(raw)

raw_filtered = raw.copy().filter(l_freq=0.5, h_freq=40.0, picks="eeg")

seizure_intervals = seizure_annotations.get(edf_file.name)
print("File: ", edf_file.name)
print("Seizure Intervals: ", seizure_intervals)

duration = raw.n_times / raw.info["sfreq"]
window_size = 5
num_windows = int(duration // window_size)

print("Duration:", duration)
print("Number of windows:", num_windows)


Extracting EDF parameters from c:\seizure-detection-ML-project\data\raw\chb01\chb01_01.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
<RawEDF | chb01_01.edf, 23 x 921600 (3600.0 s), ~161.7 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1691 samples (6.605 s)



C:\Users\rahma\AppData\Local\Temp\ipykernel_7516\4250050679.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_file, preload=True)


File:  chb01_01.edf
Seizure Intervals:  []
Duration: 3600.0
Number of windows: 720


In [129]:
windows = []
for i in range(num_windows):
    start_time = i * window_size
    end_time = start_time + window_size

    windows.append({
        "window_index": i,
        "start_time": start_time,
        "end_time": end_time
    })



print(windows[:3])
print(windows[-1])

[{'window_index': 0, 'start_time': 0, 'end_time': 5}, {'window_index': 1, 'start_time': 5, 'end_time': 10}, {'window_index': 2, 'start_time': 10, 'end_time': 15}]
{'window_index': 719, 'start_time': 3595, 'end_time': 3600}


In [130]:
for window in windows:
    start_time = window["start_time"]
    end_time = window["end_time"]

    full_seizure = any(
        start_time >= seizure_start and end_time <= seizure_end
        for seizure_start, seizure_end in seizure_intervals
    )

    overlaps_seizure = any(
        (start_time < seizure_end and end_time > seizure_start)
        for seizure_start, seizure_end in seizure_intervals
    )

    if full_seizure:
        label = "Seizure"
    elif overlaps_seizure:
        label = "Partial Seizure"
    else:
        label = "Non Seizure"

    window["label"] = label

print(windows[:3])
print(windows[-1])



[{'window_index': 0, 'start_time': 0, 'end_time': 5, 'label': 'Non Seizure'}, {'window_index': 1, 'start_time': 5, 'end_time': 10, 'label': 'Non Seizure'}, {'window_index': 2, 'start_time': 10, 'end_time': 15, 'label': 'Non Seizure'}]
{'window_index': 719, 'start_time': 3595, 'end_time': 3600, 'label': 'Non Seizure'}


In [131]:
ns = sum(
    1
    for window in windows
    if window["label"] == "Non Seizure"
)

ps = sum(
    1
    for window in windows
    if window["label"] == "Partial Seizure"
)

s = sum(
    1
    for window in windows
    if window["label"] == "Seizure"
)

print("Non Seizure Count: ", ns)
print("Partial Seizure Count: ", ps)
print("Seizure Count: ", s)

Non Seizure Count:  720
Partial Seizure Count:  0
Seizure Count:  0


In [132]:
for win in windows[597:609]:
    print(win)

{'window_index': 597, 'start_time': 2985, 'end_time': 2990, 'label': 'Non Seizure'}
{'window_index': 598, 'start_time': 2990, 'end_time': 2995, 'label': 'Non Seizure'}
{'window_index': 599, 'start_time': 2995, 'end_time': 3000, 'label': 'Non Seizure'}
{'window_index': 600, 'start_time': 3000, 'end_time': 3005, 'label': 'Non Seizure'}
{'window_index': 601, 'start_time': 3005, 'end_time': 3010, 'label': 'Non Seizure'}
{'window_index': 602, 'start_time': 3010, 'end_time': 3015, 'label': 'Non Seizure'}
{'window_index': 603, 'start_time': 3015, 'end_time': 3020, 'label': 'Non Seizure'}
{'window_index': 604, 'start_time': 3020, 'end_time': 3025, 'label': 'Non Seizure'}
{'window_index': 605, 'start_time': 3025, 'end_time': 3030, 'label': 'Non Seizure'}
{'window_index': 606, 'start_time': 3030, 'end_time': 3035, 'label': 'Non Seizure'}
{'window_index': 607, 'start_time': 3035, 'end_time': 3040, 'label': 'Non Seizure'}
{'window_index': 608, 'start_time': 3040, 'end_time': 3045, 'label': 'Non Se

In [133]:
usable_windows = [
    window
    for window in windows
    if window["label"] != "Partial Seizure"
]

print("Total windows:", len(windows))
print("Usable windows:", len(usable_windows))

Total windows: 720
Usable windows: 720


In [134]:
non_seizure_count = sum(
    1 for window in usable_windows
    if window["label"] == "Non Seizure"
)

seizure_count = sum(
    1 for window in usable_windows
    if window["label"] == "Seizure"
)

print("Non Seizure:", non_seizure_count)
print("Seizure:", seizure_count)
print("Total:", len(usable_windows))

Non Seizure: 720
Seizure: 0
Total: 720


In [135]:
window = usable_windows[0]
start_sample = int(window["start_time"] * raw_filtered.info["sfreq"])
end_sample = int(window["end_time"] * raw_filtered.info["sfreq"])
data, times = raw_filtered.get_data(picks=["FP1-F7"], start=start_sample, stop=end_sample, return_times=True)
print(window)
print(data.shape)

{'window_index': 0, 'start_time': 0, 'end_time': 5, 'label': 'Non Seizure'}
(1, 1280)


In [136]:
window_uv = data[0] * 1e6  # Convert to microvolts

print(window_uv.shape)
print(window_uv[:5])

(1280,)
[4.82808780e-14 9.14333300e+01 1.51120648e+02 1.67493104e+02
 1.53854191e+02]


In [137]:
from scipy.signal import welch

def extract_features(window_uv, sfreq):
    # Time-domain features
    mean_uv = np.mean(window_uv)
    std_uv = np.std(window_uv)
    rms_uv = np.sqrt(np.mean(window_uv ** 2))
    ptp_uv = np.ptp(window_uv)

    # Frequency-domain representation using Welch PSD
    freqs, psd_values = welch(
        window_uv,
        fs=sfreq,
        nperseg=512,
        noverlap=256
    )

    # Keep only 0.5–40 Hz
    freq_mask = (freqs >= 0.5) & (freqs <= 40)

    freqs_filtered = freqs[freq_mask]
    psd_filtered = psd_values[freq_mask]

    # Delta: 0.5–4 Hz
    delta_mask = (freqs_filtered >= 0.5) & (freqs_filtered <= 4)
    delta_power = np.trapezoid(
        psd_filtered[delta_mask],
        freqs_filtered[delta_mask]
    )

    # Theta: 4–8 Hz
    theta_mask = (freqs_filtered >= 4) & (freqs_filtered <= 8)
    theta_power = np.trapezoid(
        psd_filtered[theta_mask],
        freqs_filtered[theta_mask]
    )

    # Alpha: 8–13 Hz
    alpha_mask = (freqs_filtered >= 8) & (freqs_filtered <= 13)
    alpha_power = np.trapezoid(
        psd_filtered[alpha_mask],
        freqs_filtered[alpha_mask]
    )

    # Beta: 13–30 Hz
    beta_mask = (freqs_filtered >= 13) & (freqs_filtered <= 30)
    beta_power = np.trapezoid(
        psd_filtered[beta_mask],
        freqs_filtered[beta_mask]
    )

    # Gamma: 30–40 Hz
    gamma_mask = (freqs_filtered >= 30) & (freqs_filtered <= 40)
    gamma_power = np.trapezoid(
        psd_filtered[gamma_mask],
        freqs_filtered[gamma_mask]
    )

    return {
        "mean_uv": mean_uv,
        "std_uv": std_uv,
        "rms_uv": rms_uv,
        "ptp_uv": ptp_uv,
        "delta_power": delta_power,
        "theta_power": theta_power,
        "alpha_power": alpha_power,
        "beta_power": beta_power,
        "gamma_power": gamma_power
    }

In [138]:
sfreq = raw_filtered.info["sfreq"]
test_features = extract_features(window_uv, sfreq)
print(test_features)

{'mean_uv': np.float64(18.36955494715547), 'std_uv': np.float64(56.54156635981789), 'rms_uv': np.float64(59.450729813672226), 'ptp_uv': np.float64(317.832017227693), 'delta_power': np.float64(1099.7846064497703), 'theta_power': np.float64(265.97338762080926), 'alpha_power': np.float64(97.24328882230068), 'beta_power': np.float64(374.7500962812505), 'gamma_power': np.float64(166.4939420853371)}


In [139]:
feature_rows = []

for window in usable_windows: #first 3 rows only to test
    start_time = window["start_time"]
    end_time = window["end_time"]
    start_sample = int(start_time * raw_filtered.info["sfreq"])
    end_sample = int(end_time * raw_filtered.info["sfreq"])
    window_data = raw_filtered.get_data(
        picks=["FP1-F7"],
        start=start_sample,
        stop=end_sample
    )
    window_uv = window_data[0] * 1e6
    features = extract_features(window_uv, sfreq)
    features["patient_id"] = edf_file.parent.name
    features["recording_id"] = edf_file.stem
    features["window_index"] = window["window_index"]
    features["start_time"] = window["start_time"]
    features["end_time"] = window["end_time"]
    features["label"] = window["label"]
    feature_rows.append(features)

print(feature_rows[0])
print(features)
print(len(feature_rows))
print(feature_rows[0])

{'mean_uv': np.float64(18.36955494715547), 'std_uv': np.float64(56.54156635981789), 'rms_uv': np.float64(59.450729813672226), 'ptp_uv': np.float64(317.832017227693), 'delta_power': np.float64(1099.7846064497703), 'theta_power': np.float64(265.97338762080926), 'alpha_power': np.float64(97.24328882230068), 'beta_power': np.float64(374.7500962812505), 'gamma_power': np.float64(166.4939420853371), 'patient_id': 'chb01', 'recording_id': 'chb01_01', 'window_index': 0, 'start_time': 0, 'end_time': 5, 'label': 'Non Seizure'}
{'mean_uv': np.float64(-3.3609322465654374), 'std_uv': np.float64(44.22111398110532), 'rms_uv': np.float64(44.348650343566405), 'ptp_uv': np.float64(483.98769877801533), 'delta_power': np.float64(1992.95783225793), 'theta_power': np.float64(470.35039332724796), 'alpha_power': np.float64(87.33252352962306), 'beta_power': np.float64(138.9160159605051), 'gamma_power': np.float64(77.3768401375703), 'patient_id': 'chb01', 'recording_id': 'chb01_01', 'window_index': 719, 'start_

In [140]:
features_df = pd.DataFrame(feature_rows)
print(features_df["label"].value_counts())

label
Non Seizure    720
Name: count, dtype: int64


In [141]:
header = edf_file.stem
processed_path = (
project_root / "data" / "processed" / f"{header}_features.csv"
)

features_df.to_csv(processed_path, index=False)

print("Saved:", processed_path.exists())
print("File: ", processed_path.name)

Saved: True
File:  chb01_01_features.csv
